In [1]:
import pandas as pd
import os

In [466]:
# Function to calculate cost
def calculate_cost(row):
    # Base cost per km
    cost = row['Distance [km]'] * cost_factor_transportation
    
    # Add additional costs for Suez or Panama
    if pd.notna(row['Suez or Panama']):
        if 'Suez' in row['Suez or Panama']:
            cost += cost_factor_suez
        elif 'Panama' in row['Suez or Panama']:
            cost += cost_factor_panama
    
    return cost

# Function to create the new dataframe based on the "From" column for liquefaction cost
def create_LNG_liquefaction_df(df, factor):
    # Step 1: Create a new dataframe with unique "From" values and the corresponding "To" column
    unique_from_df = pd.DataFrame(df['From'].unique(), columns=['From'])
    
    # Step 2: Adjust the "From" column by removing "_LNG" and place the original "From" values in the "To" column
    unique_from_df['To'] = unique_from_df['From']
    unique_from_df['From'] = unique_from_df['From'].str.replace('_LNG', '')
    
    # Step 3: Add the "Cost" column with the value of cost_factor_liquification
    unique_from_df['Cost'] = factor
    
    return unique_from_df

# Function to create the new dataframe based on the "To" column for regasification cost
def create_LNG_regasification_df(df, factor):
    # Step 1: Create a new dataframe with unique "To" values and the corresponding "To" column
    unique_from_df = pd.DataFrame(df['To'].unique(), columns=['To'])
    
    # Step 2: Adjust the "To" column by removing "_LNG" and place the original "To" values in the "From" column
    unique_from_df['From'] = unique_from_df['To']
    unique_from_df['To'] = unique_from_df['To'].str.replace('_LNG', '')
    
    # Step 3: Add the "Cost" column with the value of cost_factor_liquification
    unique_from_df['Cost'] = factor
    
    return unique_from_df

#Define a function to multiply the distance by a cost factor for the pipelines
def pipeline_transport_cost(df, cost_factor):
    df["Cost"] = df["distance [km]"] * cost_factor
    return df

# Function to create df_supply_demand_global
def create_supply_demand_df(df, supply=True):
    # Create the new dataframe with required columns
    df_supply_demand_global = pd.DataFrame({
        'Commodity': ['Methane'] * len(df),  # Set 'Methane' for all rows
        'Node': df['Country'] + ('_Prod' if supply else ''),  # Append '_Prod' if supply is True
        'Supply': df['GWh [2020]']  # Use 'GWh [2020]' for supply
    })
    
    return df_supply_demand_global

def expand_with_hydrogen(df, hydrogen_investment):
    if hydrogen_investment:
        return df  # If investment is allowed, return the original dataframe unchanged

    # Check if the input dataframe follows the (Commodity, Source, Destination) structure
    if {'Commodity', 'Source', 'Destination'}.issubset(df.columns):
        # Copy relevant columns and create a hydrogen version
        hydrogen_df = df[['Commodity', 'Source', 'Destination']].copy()
        hydrogen_df['Commodity'] = 'Hydrogen'  # Replace Commodity with Hydrogen
        
    # If the input dataframe follows the (Commodity, Node, Supply) structure
    elif {'Commodity', 'Node', 'Supply'}.issubset(df.columns):
        # Copy relevant columns and create a hydrogen version
        hydrogen_df = df[['Commodity', 'Node']].copy()
        hydrogen_df['Commodity'] = 'Hydrogen'  # Replace Commodity with Hydrogen
    
    else:
        raise ValueError("Unexpected dataframe format. Must contain either ['Commodity', 'Source', 'Destination'] or ['Commodity', 'Node', 'Supply']")

    # Fill all other columns with 0
    for col in df.columns:
        if col not in hydrogen_df.columns:  # Skip the required columns
            hydrogen_df[col] = 0

    # Combine the original dataframe with the new hydrogen dataframe
    df_expanded = pd.concat([df, hydrogen_df], ignore_index=True)

    return df_expanded

def integrate_pipeline_costs(df_pipelines, df_transport_cost):
    """
    Merges transport cost data into the pipeline dataframe based on matching From-To relationships, 
    considering both directions (From->To and To->From).
    
    Parameters:
        df_pipelines (pd.DataFrame): DataFrame containing pipeline capacities.
        df_transport_cost (pd.DataFrame): DataFrame containing transport distances and costs.
        
    Returns:
        pd.DataFrame: Updated pipeline DataFrame with an additional 'cost' column.
    """
    # Create reversed pairs for bidirectional matching (From -> To and To -> From)
    df_reversed = df_transport_cost.rename(columns={'From': 'To', 'To': 'From', 'Cost': 'Cost_reversed'})
    
    # Concatenate original and reversed cost data to handle both directions
    df_cost = pd.concat([df_transport_cost[['From', 'To', 'Cost']], df_reversed[['From', 'To', 'Cost_reversed']]], ignore_index=True)
    
    # Merge the concatenated cost data with df_pipelines to get the corresponding cost
    df_pipelines = df_pipelines.merge(
        df_cost, 
        on=['From', 'To'], 
        how='left'
    )
    
    # For cases where the reverse relation exists, use the reversed cost value
    df_pipelines['Cost'] = df_pipelines['Cost'].fillna(df_pipelines['Cost_reversed'])

    # Drop the extra reversed cost column (no longer needed)
    df_pipelines = df_pipelines.drop(columns=['Cost_reversed'])
    
    return df_pipelines

In [467]:
def create_edges_cap_cost_dataframe(df_cost_per_km, LNG_regasification_df, LNG_liquefaction_df, 
                                    df_global_pipe_transport_cost, df_pipelines_europe, df_LNG_europe):
    # Step 1: Collect unique From-To pairs from all dataframes
    unique_pairs = set()

    # Collecting unique pairs from each dataframe
    for df in [df_cost_per_km, LNG_regasification_df, LNG_liquefaction_df, 
               df_global_pipe_transport_cost, df_pipelines_europe, df_LNG_europe]:
        for _, row in df.iterrows():
            unique_pairs.add((row['From'], row['To']))

    # Step 2: Create the new structured dataframe with Source and Destination columns
    df_all_cost = pd.DataFrame(unique_pairs, columns=['Source', 'Destination'])

    # Step 3: Add required columns
    df_all_cost.insert(0, 'Commodity', 'Methane')  # Set commodity as Methane

    # Step 4: Merge cost and capacity data

    # Merge with df_cost_per_km (Cost)
    df_all_cost = df_all_cost.merge(
        df_cost_per_km[['From', 'To', 'Cost']],
        left_on=['Source', 'Destination'],
        right_on=['From', 'To'],
        how='left'
    ).drop(columns=['From', 'To']).rename(columns={'Cost': 'Cost_cost_per_km'})

    # Merge with LNG_regasification_df (Cost)
    df_all_cost = df_all_cost.merge(
        LNG_regasification_df[['From', 'To', 'Cost']],
        left_on=['Source', 'Destination'],
        right_on=['From', 'To'],
        how='left'
    ).drop(columns=['From', 'To']).rename(columns={'Cost': 'Cost_LNG_regasification'})

    # Merge with LNG_liquefaction_df (Cost)
    df_all_cost = df_all_cost.merge(
        LNG_liquefaction_df[['From', 'To', 'Cost']],
        left_on=['Source', 'Destination'],
        right_on=['From', 'To'],
        how='left'
    ).drop(columns=['From', 'To']).rename(columns={'Cost': 'Cost_LNG_liquefaction'})

    # Merge with df_global_pipe_transport_cost (Cost)
    df_all_cost = df_all_cost.merge(
        df_global_pipe_transport_cost[['From', 'To', 'Cost']],
        left_on=['Source', 'Destination'],
        right_on=['From', 'To'],
        how='left'
    ).drop(columns=['From', 'To']).rename(columns={'Cost': 'Cost_global_pipe_transport'})

    # Merge with df_pipelines_europe (Cost and GWh/a)
    df_all_cost = df_all_cost.merge(
        df_pipelines_europe[['From', 'To', 'Cost', 'GWh/a']],
        left_on=['Source', 'Destination'],
        right_on=['From', 'To'],
        how='left'
    ).drop(columns=['From', 'To']).rename(columns={'Cost': 'Cost_pipelines_europe', 'GWh/a': 'GWh_a_pipelines_europe'})

    # Merge with df_LNG_europe (Cost and GWh/a)
    df_all_cost = df_all_cost.merge(
        df_LNG_europe[['From', 'To', 'Cost', 'GWh/a']],
        left_on=['Source', 'Destination'],
        right_on=['From', 'To'],
        how='left'
    ).drop(columns=['From', 'To']).rename(columns={'Cost': 'Cost_LNG_europe', 'GWh/a': 'GWh_a_LNG_europe'})

    # Step 5: Set costs_edge (sum available cost values)
    df_all_cost['costs_edge'] = df_all_cost[
        ['Cost_cost_per_km', 'Cost_LNG_regasification', 'Cost_LNG_liquefaction', 
         'Cost_global_pipe_transport', 'Cost_pipelines_europe', 'Cost_LNG_europe']
    ].sum(axis=1, min_count=1)

    # Step 6: Set correct capacities
    # If pipeline/LNG capacity exists, use it. Otherwise, keep 9999.
    df_all_cost['initial_capacities'] = df_all_cost['GWh_a_pipelines_europe'].fillna(0) + df_all_cost['GWh_a_LNG_europe'].fillna(0)
    df_all_cost['max_capacities'] = df_all_cost['initial_capacities']

    # If capacity is still 0 (no data), set to 9999
    df_all_cost.loc[df_all_cost['initial_capacities'] == 0, 'initial_capacities'] = 9999
    df_all_cost.loc[df_all_cost['max_capacities'] == 0, 'max_capacities'] = 9999

    # Step 7: Drop unnecessary columns
    df_all_cost = df_all_cost.drop(columns=[
        'Cost_cost_per_km', 'Cost_LNG_regasification', 'Cost_LNG_liquefaction', 
        'Cost_global_pipe_transport', 'Cost_pipelines_europe', 'Cost_LNG_europe',
        'GWh_a_pipelines_europe', 'GWh_a_LNG_europe'
    ])

    # Step 8: Add empty columns for future values
    df_all_cost['new_build_cost'] = 1000000
    df_all_cost['conversion_cost'] = 0
    df_all_cost['conversion_capacity_factor'] = 1

    return df_all_cost

def merge_all_edges(*dfs):
    """
    Merges multiple edge dataframes while removing duplicate Source-Destination pairs.
    
    Args:
        *dfs: Any number of dataframes to be merged.
    
    Returns:
        A merged dataframe with duplicates removed.
    """
    # Step 1: Concatenate all dataframes
    df_merged = pd.concat(dfs, ignore_index=True)

    # Step 2: Remove duplicate edges (keeping the first occurrence)
    df_merged = df_merged.drop_duplicates(subset=['Source', 'Destination'], keep='first')

    return df_merged

In [498]:
def create_base_edges(df):
    """Create a dataframe with unique (From, To) pairs."""
    unique_pairs = set((row['From'], row['To']) for _, row in df.iterrows())
    return pd.DataFrame(unique_pairs, columns=['Source', 'Destination'])

#function to generate structure of European pipeline parameters
def process_european_pipeline_edges(df_pipelines_europe):
    """Process European pipeline transport cost and capacity data."""
    df_pipelines = create_base_edges(df_pipelines_europe)

    # Merge cost & capacity
    df_pipelines = df_pipelines.merge(
        df_pipelines_europe[['From', 'To', 'Cost', 'GWh/a']],
        left_on=['Source', 'Destination'],
        right_on=['From', 'To'],
        how='left'
    ).drop(columns=['From', 'To']).rename(columns={'Cost': 'costs_edge', 'GWh/a': 'Capacity_pipelines'})

    # Default values for missing capacities
    df_pipelines['initial_capacities'] = df_pipelines['Capacity_pipelines'].fillna(9999)
    df_pipelines['max_capacities'] = df_pipelines['initial_capacities']
    df_pipelines = df_pipelines.drop(columns=['Capacity_pipelines'])

    # Add default columns
    df_pipelines.insert(0, 'Commodity', 'Methane')
    df_pipelines['new_build_cost'] = 1000000
    df_pipelines['conversion_cost'] = 0
    df_pipelines['conversion_capacity_factor'] = 1

    column_to_move = df_pipelines.pop("costs_edge")

    # insert column with insert(location, column_name, column_value)
    df_pipelines.insert(5, "costs_edge", column_to_move)

    return df_pipelines

#process LNG import function (for European import capacities)
def process_LNG_import_edges(df_LNG_europe):
    """Process LNG import terminal costs and capacities."""
    df_LNG = create_base_edges(df_LNG_europe)

    df_LNG = df_LNG.merge(
        df_LNG_europe[['From', 'To', 'Cost', 'GWh/a']],
        left_on=['Source', 'Destination'],
        right_on=['From', 'To'],
        how='left'
    ).drop(columns=['From', 'To']).rename(columns={'Cost': 'costs_edge', 'GWh/a': 'Capacity_LNG_import'})

    df_LNG['initial_capacities'] = df_LNG['Capacity_LNG_import'].fillna(9999)
    df_LNG['max_capacities'] = df_LNG['initial_capacities']
    df_LNG = df_LNG.drop(columns=['Capacity_LNG_import'])

    # Add default columns
    df_LNG.insert(0, 'Commodity', 'Methane')
    df_LNG['new_build_cost'] = 1000000
    df_LNG['conversion_cost'] = 0
    df_LNG['conversion_capacity_factor'] = 1

    column_to_move = df_LNG.pop("costs_edge")

    # insert column with insert(location, column_name, column_value)
    df_LNG.insert(5, "costs_edge", column_to_move)

    return df_LNG

#function to process LNG global exchange parameters for the edges
def process_LNG_global_edges(df_LNG_global):
    """Process global LNG transport costs and capacities."""
    df_LNG_global_processed = create_base_edges(df_LNG_global)

    # Merge cost and distance data
    df_LNG_global_processed = df_LNG_global_processed.merge(
        df_LNG_global[['From', 'To', 'Cost', 'GWh/a']],
        left_on=['Source', 'Destination'],
        right_on=['From', 'To'],
        how='left'
    ).drop(columns=['From', 'To']).rename(columns={'Cost': 'costs_edge', 'GWh/a': 'Capacity_LNG_global'})

    # Handle capacity values
    df_LNG_global_processed['initial_capacities'] = df_LNG_global_processed['Capacity_LNG_global'].fillna(9999)
    df_LNG_global_processed['max_capacities'] = df_LNG_global_processed['initial_capacities']
    df_LNG_global_processed = df_LNG_global_processed.drop(columns=['Capacity_LNG_global'])

    # Add default columns
    df_LNG_global_processed.insert(0, 'Commodity', 'Methane')
    df_LNG_global_processed['new_build_cost'] = 1000000
    df_LNG_global_processed['conversion_cost'] = 0
    df_LNG_global_processed['conversion_capacity_factor'] = 1

    # Move `costs_edge` column to the correct position
    column_to_move = df_LNG_global_processed.pop("costs_edge")
    df_LNG_global_processed.insert(5, "costs_edge", column_to_move)

    return df_LNG_global_processed


def process_LNG_regasification_edges(df_LNG_regasification):
    """Process LNG regasification costs."""
    df_regas = create_base_edges(df_LNG_regasification)

    df_regas = df_regas.merge(
        df_LNG_regasification[['From', 'To', 'Cost']],
        left_on=['Source', 'Destination'],
        right_on=['From', 'To'],
        how='left'
    ).drop(columns=['From', 'To']).rename(columns={'Cost': 'costs_edge'})

        # Default values for missing capacities
    df_regas['initial_capacities'] = 9999
    df_regas['max_capacities'] = df_regas['initial_capacities']

    # Add default columns
    df_regas.insert(0, 'Commodity', 'Methane')
    df_regas['new_build_cost'] = 1000000
    df_regas['conversion_cost'] = 0
    df_regas['conversion_capacity_factor'] = 1

    column_to_move = df_regas.pop("costs_edge")

    # insert column with insert(location, column_name, column_value)
    df_regas.insert(5, "costs_edge", column_to_move)
    
    return df_regas


#function to create parameters for liquification nodes
def process_LNG_liquefaction_edges(df_LNG_liquefaction):
    """Process LNG liquefaction costs."""
    df_liquefaction = create_base_edges(df_LNG_liquefaction)

    df_liquefaction = df_liquefaction.merge(
        df_LNG_liquefaction[['From', 'To', 'Cost']],
        left_on=['Source', 'Destination'],
        right_on=['From', 'To'],
        how='left'
    ).drop(columns=['From', 'To']).rename(columns={'Cost': 'costs_edge'})

    # Default values for missing capacities
    df_liquefaction['initial_capacities'] = 9999
    df_liquefaction['max_capacities'] = df_liquefaction['initial_capacities']

    # Add default columns
    df_liquefaction.insert(0, 'Commodity', 'Methane')
    df_liquefaction['new_build_cost'] = 1000000
    df_liquefaction['conversion_cost'] = 0
    df_liquefaction['conversion_capacity_factor'] = 1

    column_to_move = df_liquefaction.pop("costs_edge")

    # insert column with insert(location, column_name, column_value)
    df_liquefaction.insert(5, "costs_edge", column_to_move)

    return df_liquefaction

#function to create parameters for global pipelines
def process_global_pipeline_edges(df_global_pipe_transport_cost):
    """Process global pipeline transport costs."""
    df_global_pipeline = create_base_edges(df_global_pipe_transport_cost)

    df_global_pipeline = df_global_pipeline.merge(
        df_global_pipe_transport_cost[['From', 'To', 'Cost', 'GWh/a']],
        left_on=['Source', 'Destination'],
        right_on=['From', 'To'],
        how='left'
    ).drop(columns=['From', 'To']).rename(columns={'Cost': 'costs_edge', 'GWh/a': 'Capacity_pipelines'})

    # Default values for missing capacities
    df_global_pipeline['initial_capacities'] = df_global_pipeline['Capacity_pipelines'].fillna(9999)
    df_global_pipeline['max_capacities'] = df_global_pipeline['initial_capacities']
    df_global_pipeline = df_global_pipeline.drop(columns=['Capacity_pipelines'])

    # Add default columns
    df_global_pipeline.insert(0, 'Commodity', 'Methane')
    df_global_pipeline['new_build_cost'] = 1000000
    df_global_pipeline['conversion_cost'] = 0
    df_global_pipeline['conversion_capacity_factor'] = 1

    column_to_move = df_global_pipeline.pop("costs_edge")

    # insert column with insert(location, column_name, column_value)
    df_global_pipeline.insert(5, "costs_edge", column_to_move)

    return df_global_pipeline

def remove_existing_LNG_edges(df_regasification_LNG_edges, df_europe_LNG_edges):
    """Removes rows from df_regasification_LNG_edges if the Source-Destination pair exists in df_europe_LNG_edges."""
    existing_edges = set(zip(df_europe_LNG_edges['Source'], df_europe_LNG_edges['Destination']))
    
    df_filtered = df_regasification_LNG_edges[
        ~df_regasification_LNG_edges.apply(lambda row: (row['Source'], row['Destination']) in existing_edges, axis=1)
    ]
    
    return df_filtered

In [469]:
hydrogen_investment=False

cost_factor_transportation = 2
cost_factor_liquefaction = 2000
cost_factor_regasification = 2000
cost_factor_panama = 2000
cost_factor_suez = 2000

cost_factor_pipelines = 0.5

## Import Data

In [470]:
# Specify the path to your Excel file
input_file_path_1 = os.path.join('..', '..','01_data', '01_input_data', '02_processed')
excel_file_name = '\\Gas_Import_Russian_Invasion.xlsx'
# Specify the path to your Excel file
input_file_path_2 = os.path.join('..', '..','00_code_base', '07_data_prep')
distances_file_name = '\\distances.xlsx'

input_file_path_1  = input_file_path_1 + excel_file_name
full_input_path_1 = os.path.abspath(os.path.join(os.getcwd(), input_file_path_1))
input_file_path_2  = input_file_path_2 + distances_file_name
full_input_path_2 = os.path.abspath(os.path.join(os.getcwd(), input_file_path_2))

In [471]:
df_LNG_global = pd.read_excel(full_input_path_1, sheet_name='global_LNG_connections_2020')
df_LNG_europe = pd.read_excel(full_input_path_1, sheet_name='nodes_lng_europe_2020')
df_production_europe = pd.read_excel(full_input_path_1, sheet_name='nodes_gas_prod_europe_2020')
df_consumption_europe = pd.read_excel(full_input_path_1, sheet_name='nodes_demand_europe_2020')
df_production_global = pd.read_excel(full_input_path_1, sheet_name='nodes_gas_prod_world_2020')
df_consumption_global = pd.read_excel(full_input_path_1, sheet_name='nodes_demand_world_2020')
df_pipelines_europe = pd.read_excel(full_input_path_1, sheet_name='Connections_2021')
df_pipelines_global = pd.read_excel(full_input_path_1, sheet_name='Global_Connections')

#load input for inner-European distances
df_distances = pd.read_excel(full_input_path_2, sheet_name='distances')
#Drop the "Unnamed: 0" column in df_distances
df_distances.drop(columns=["Unnamed: 0"], inplace=True)

In [472]:
#adjust the data frames and remove unnecessary content
#gobal demand
df_consumption_global = df_consumption_global.iloc[:-3]
df_consumption_global = df_consumption_global.iloc[:, :-2]
#gobal production
df_production_global = df_production_global.iloc[:-3]
#globale pipelines
df_pipelines_global = df_pipelines_global.iloc[:, :-5]
#europe demand
df_consumption_europe = df_consumption_europe.iloc[:-11]
#europe production
df_production_europe = df_production_europe.iloc[:-7]
df_production_europe = df_production_europe.iloc[:, :-7]
#europe LNG
df_LNG_europe = df_LNG_europe.iloc[:, :-6]
#add a From column to the European LNG data
df_LNG_europe.insert(1, 'From', df_LNG_europe['To'] + '_LNG')
#LNG global
df_LNG_global = df_LNG_global.iloc[:, :-5]

#adjust naming
df_distances = df_distances.rename(columns={"from [NUTS_ID]": "From"})
df_distances = df_distances.rename(columns={"to [NUTS_ID]": "To"})

### Demand and Supply input sheet

In [473]:
#demand Europe
df_demand_europe = create_supply_demand_df(df_consumption_europe, supply=False)
#supply Europe
df_supply_europe = create_supply_demand_df(df_production_europe)

In [474]:
#demand Europe
df_demand_global = create_supply_demand_df(df_consumption_global, supply=False)
#supply global
df_supply_global = create_supply_demand_df(df_production_global)

In [475]:
# Combine the demand and supply data frames
df_all_supply_demand = pd.concat([df_demand_europe, df_supply_europe, df_demand_global, df_supply_global], ignore_index=True)

In [476]:
# Apply the function to each row and create a new column 'Cost per km'
df_LNG_global['Cost'] = df_LNG_global.apply(calculate_cost, axis=1)

# Create a new dataframe with the relevant columns
df_cost_per_km = df_LNG_global[['From', 'To', 'Distance [km]', 'Suez or Panama', 'Cost']]

In [477]:
# Get LNG liquefaction cost df
LNG_liquefaction_df = create_LNG_liquefaction_df(df_LNG_global, cost_factor_liquefaction)
# Get LNG regasification cost df
LNG_regasification_df = create_LNG_regasification_df(df_LNG_global, cost_factor_regasification)

In [478]:
#add regasification cost to European LNG import nodes
df_LNG_europe.insert(3, "Cost", cost_factor_regasification)

In [479]:
#Pipeline cost
#calculate European pipeline cost
df_european_pipe_transport_cost = pipeline_transport_cost(df_distances, cost_factor_pipelines)
df_global_pipe_transport_cost = pipeline_transport_cost(df_pipelines_global, cost_factor_pipelines)

In [480]:
# Example usage:
df_pipelines_europe = integrate_pipeline_costs(df_pipelines_europe, df_european_pipe_transport_cost)

In [481]:
df_europe_pipe_edges = process_european_pipeline_edges(df_pipelines_europe)

In [482]:
df_global_pipe_edges = process_global_pipeline_edges(df_global_pipe_transport_cost)
df_liquefaction_LNG_edges = process_LNG_liquefaction_edges(LNG_liquefaction_df)
df_regasification_LNG_edges = process_LNG_regasification_edges(LNG_regasification_df)
df_europe_LNG_edges = process_LNG_import_edges(df_LNG_europe)
df_europe_pipe_edges = process_european_pipeline_edges(df_pipelines_europe)
df_LNG_global_edges = process_LNG_global_edges(df_LNG_global)
#reomve duplicates from the regasification edges df
df_regasification_LNG_edges = remove_existing_LNG_edges(df_regasification_LNG_edges, df_europe_LNG_edges)

In [493]:
# Apply the create_edges_cap_cost_dataframe function to get edges input 
df_edges_cap_cost = merge_all_edges(
    df_europe_pipe_edges, 
    df_europe_LNG_edges, 
    df_regasification_LNG_edges, 
    df_liquefaction_LNG_edges, 
    df_global_pipe_edges, 
    df_LNG_global_edges
)

### check for hydrogen and append with zeros if no repurpose investigation

In [494]:
# Example: Calling function with hydrogen_investment = False
df_edges_complete = expand_with_hydrogen(df_edges_cap_cost, hydrogen_investment)

# Display the result
df_edges_complete

,Commodity,Source,Destination,initial_capacities,max_capacities,costs_edge,new_build_cost,conversion_cost,conversion_capacity_factor
0,Methane,LY,IT,173667.000000,173667.000000,917.630501,1000000,0,1
1,Methane,UK,BE,82928.000000,82928.000000,302.582951,1000000,0,1
2,Methane,UA,RO,73584.000000,73584.000000,301.662316,1000000,0,1
3,Methane,LV,RU,31025.000000,31025.000000,1771.987906,1000000,0,1
4,Methane,BY,UA,12969.074413,12969.074413,276.978424,1000000,0,1
...,...,...,...,...,...,...,...,...,...
653,Hydrogen,EG_LNG,CN_LNG,0.000000,0.000000,0.000000,0,0,0
654,Hydrogen,RU_LNG,PL_LNG,0.000000,0.000000,0.000000,0,0,0
655,Hydrogen,ME_LNG,CN_LNG,0.000000,0.000000,0.000000,0,0,0
656,Hydrogen,RU_LNG,AS_LNG,0.000000,0.000000,0.000000,0,0,0


In [495]:
# Example: Calling function with hydrogen_investment = False
df_demand_supply_complete = expand_with_hydrogen(df_all_supply_demand, hydrogen_investment)

# Display the result
df_demand_supply_complete

,Commodity,Node,Supply
0,Methane,AL,-7717.933443
1,Methane,AT,-83311.215033
2,Methane,BE,-166072.218600
3,Methane,BA,-9469.790436
4,Methane,BG,-28534.750500
...,...,...,...
231,Hydrogen,IN_Prod,0.000000
232,Hydrogen,AS_Prod,0.000000
233,Hydrogen,CR_Prod,0.000000
234,Hydrogen,EG_Prod,0.000000


# export

In [497]:
df_edges_complete.to_excel("inputs.xlsx", index=False, sheet_name='Parameters')